# 02 - Data Wrangling: Duplicate Transaction Detection

**Project:** Credit card fraud prediction MVP.

**What this notebook does:** finds two kinds of duplicate-looking activity in the
transaction log, measures how much money each one represents, and writes a
cleaned dataset for the modeling step.

The two patterns:

1. **Reversed transactions.** A purchase that is later undone by a reversal for
   the same account, merchant, and amount. The two entries net to zero, so
   counting both would overstate revenue.
2. **Multi-swipe transactions.** The same card charged the same amount at the
   same merchant two or more times within a few minutes. This is almost always a
   point-of-sale terminal retrying after a failed read, not a customer buying the
   same thing twice.

Neither pattern is fraud. Both are data-quality issues that would distort any
revenue number or model built on the raw log.


## 1. Load the data

Loads the Parquet cache written by `01_eda.ipynb`. If it is missing, run that
notebook first (it downloads and parses the raw file).


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = RAW_DIR / "transactions.parquet"
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"{CACHE_PATH} not found. Run notebooks/01_eda.ipynb first to build it."
    )

df = pd.read_parquet(CACHE_PATH)
df["transactionDateTime"] = pd.to_datetime(df["transactionDateTime"])
df = df.sort_values("transactionDateTime").reset_index(drop=True)
df["row_id"] = np.arange(len(df))          # stable internal key for this notebook
N_START = len(df)
print(f"Loaded {N_START:,} transactions")
print(df["transactionType"].value_counts(dropna=False))


Loaded 786,363 transactions
transactionType
PURCHASE                745193
REVERSAL                 20303
ADDRESS_VERIFICATION     20169
                           698
Name: count, dtype: int64


In [2]:
# The three fields that define "the same transaction" for both checks.
MATCH_KEYS = ["accountNumber", "merchantName", "transactionAmount"]
df[MATCH_KEYS + ["transactionDateTime", "transactionType"]].head()


,accountNumber,merchantName,transactionAmount,transactionDateTime,transactionType
0,419104777,Washington Post,44.09,2016-01-01 00:01:02,PURCHASE
1,674577133,staples.com,329.57,2016-01-01 00:01:44,PURCHASE
2,958438658,cheapfast.com,164.57,2016-01-01 00:01:47,PURCHASE
3,851126461,discount.com,122.83,2016-01-01 00:02:04,PURCHASE
4,148963316,Fast Repair,0.00,2016-01-01 00:02:19,ADDRESS_VERIFICATION


## 2. Reversed transactions

**Working definition:** a `REVERSAL` row is matched to the earliest still-unpaired
`PURCHASE` row that has the same account, merchant, and amount and occurred at or
before the reversal. Each matched pair is removed from the dataset, because the
purchase and its reversal cancel out.

Reversals that cannot be matched to a prior purchase are kept, but flagged with a
`reversalUnmatched` column so the modeling step can see them. They are a small
minority and may be timing artifacts in this synthetic data.


In [3]:
# Restrict to the purchase/reversal rows in groups that actually contain a
# reversal - no need to scan the whole table.
pr = df[df["transactionType"].isin(["PURCHASE", "REVERSAL"])].copy()
rev_group_keys = pr.loc[pr["transactionType"] == "REVERSAL", MATCH_KEYS].drop_duplicates()
pr = pr.merge(rev_group_keys, on=MATCH_KEYS, how="inner")
print(f"{len(rev_group_keys):,} (account, merchant, amount) groups contain a reversal")
print(f"{len(pr):,} purchase/reversal rows fall in those groups")


20,270 (account, merchant, amount) groups contain a reversal
38,436 purchase/reversal rows fall in those groups


In [4]:
# Greedy one-to-one pairing within each group, ordered by time.
# TODO(refine): this Python loop is clear but slow; vectorize with a merge_asof
# on time within each group if the dataset grows.
reversed_purchase_ids: set[int] = set()
reversed_reversal_ids: set[int] = set()

for _, g in pr.groupby(MATCH_KEYS, sort=False):
    g = g.sort_values("transactionDateTime")
    purchases = g.loc[g["transactionType"] == "PURCHASE", ["row_id", "transactionDateTime"]]
    purchases = list(purchases.itertuples(index=False, name=None))
    taken = [False] * len(purchases)
    for rev in g.loc[g["transactionType"] == "REVERSAL"].itertuples(index=False):
        for i, (pid, ptime) in enumerate(purchases):
            if not taken[i] and ptime <= rev.transactionDateTime:
                taken[i] = True
                reversed_purchase_ids.add(pid)
                reversed_reversal_ids.add(rev.row_id)
                break

n_reversed_pairs = len(reversed_reversal_ids)
reversed_dollars = df.loc[df["row_id"].isin(reversed_purchase_ids), "transactionAmount"].sum()
n_reversals_total = int((df["transactionType"] == "REVERSAL").sum())
n_reversals_unmatched = n_reversals_total - n_reversed_pairs

print(f"Matched purchase + reversal pairs : {n_reversed_pairs:,}")
print(f"Purchase dollars that net to zero : ${reversed_dollars:,.2f}")
print(f"Reversals with no prior purchase  : {n_reversals_unmatched:,} (kept, flagged)")
print(f"Rows this removes                 : {n_reversed_pairs * 2:,}")


Matched purchase + reversal pairs : 17,758
Purchase dollars that net to zero : $2,666,519.27
Reversals with no prior purchase  : 2,545 (kept, flagged)
Rows this removes                 : 35,516


In [5]:
# Show one matched pair end to end.
example_key = df.loc[df["row_id"].isin(reversed_reversal_ids), MATCH_KEYS].iloc[0]
mask = np.logical_and.reduce([df[k] == example_key[k] for k in MATCH_KEYS])
df.loc[mask, MATCH_KEYS + ["transactionDateTime", "transactionType", "isFraud"]]


,accountNumber,merchantName,transactionAmount,transactionDateTime,transactionType,isFraud
169,829756717,Auntie Anne's #274744,3.7,2016-01-01 01:51:31,PURCHASE,False
174,829756717,Auntie Anne's #274744,3.7,2016-01-01 01:54:16,REVERSAL,False


**Result - reversed transactions.** About **17,800 purchases** are undone by
a later reversal, worth roughly **$2.67 million** in charges that net to zero.
Removing both legs takes out about **35,500 rows**. A further **2,500 reversals**
have no matching earlier purchase; those are kept and flagged rather than
guessed at.


## 3. Multi-swipe transactions

**Working definition:** among the purchases left after reversal removal, sort each
(account, merchant, amount) group by time. Any purchase that lands within
**5 minutes** of the previous purchase in its group is treated as a terminal
retry. The first charge in each burst is kept; the later ones are dropped.

The 5-minute window is a starting point. The cell after the results shows how the
count changes as the window widens, to check the choice is not arbitrary.


In [6]:
WINDOW_MINUTES = 5

survivors = df[~df["row_id"].isin(reversed_purchase_ids)]
purchases = survivors[survivors["transactionType"] == "PURCHASE"].sort_values(
    MATCH_KEYS + ["transactionDateTime"]
)
gap_minutes = (
    purchases.groupby(MATCH_KEYS, sort=False)["transactionDateTime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)
is_followon = gap_minutes.ge(0) & gap_minutes.le(WINDOW_MINUTES)
multiswipe_ids = set(purchases.loc[is_followon, "row_id"])

multiswipe_dollars = df.loc[df["row_id"].isin(multiswipe_ids), "transactionAmount"].sum()
multiswipe_accounts = df.loc[df["row_id"].isin(multiswipe_ids), "accountNumber"].nunique()
print(f"Follow-on swipes removed        : {len(multiswipe_ids):,}")
print(f"Duplicated charge dollars       : ${multiswipe_dollars:,.2f}")
print(f"Accounts affected               : {multiswipe_accounts:,}")


Follow-on swipes removed        : 7,259
Duplicated charge dollars       : $1,071,670.25
Accounts affected               : 1,879


In [7]:
# Sensitivity of the count to the time window.
rows = []
for w in [1, 2, 5, 10, 15, 30]:
    m = gap_minutes.ge(0) & gap_minutes.le(w)
    rows.append(
        {
            "window_minutes": w,
            "followon_swipes": int(m.sum()),
            "dollars": round(df.loc[df["row_id"].isin(purchases.loc[m, "row_id"]), "transactionAmount"].sum(), 2),
        }
    )
pd.DataFrame(rows)


,window_minutes,followon_swipes,dollars
0,1,2411,366727.52
1,2,4813,723665.39
2,5,7259,1071670.25
3,10,7260,1071881.51
4,15,7260,1071881.51
5,30,7260,1071881.51


In [8]:
# Show one multi-swipe burst.
ms_key = df.loc[df["row_id"].isin(multiswipe_ids), MATCH_KEYS].iloc[0]
mask = np.logical_and.reduce([df[k] == ms_key[k] for k in MATCH_KEYS])
df.loc[mask, MATCH_KEYS + ["transactionDateTime", "transactionType"]].assign(
    gap_from_prev_min=lambda d: d["transactionDateTime"].diff().dt.total_seconds().div(60)
)


,accountNumber,merchantName,transactionAmount,transactionDateTime,transactionType,gap_from_prev_min
168,708054411,Wayfair.com,3.32,2016-01-01 01:51:16,PURCHASE,NaN
172,708054411,Wayfair.com,3.32,2016-01-01 01:52:37,PURCHASE,1.35


**Result - multi-swipe transactions.** About **7,300 purchases** are
repeat swipes within 5 minutes of an identical charge, worth roughly
**$1.07 million** in charges that should not have gone through twice. They touch
about **1,900 accounts**. Widening the window past 5 minutes barely changes the
count (only one extra row by 10 minutes), which says these really are tight
bursts rather than a gradual tail. Five minutes is a safe cut.


## 4. Build and save the deduped dataset

Removed: both legs of every matched purchase/reversal pair, plus every follow-on
multi-swipe. Kept: all first-of-burst purchases, all address verification checks,
and the unmatched reversals (flagged). The result is written to
`data/processed/transactions_deduped.parquet` for the modeling notebook.


In [9]:
drop_ids = reversed_purchase_ids | reversed_reversal_ids | multiswipe_ids

clean = df[~df["row_id"].isin(drop_ids)].copy()
clean["reversalUnmatched"] = (clean["transactionType"] == "REVERSAL") & (
    ~clean["row_id"].isin(reversed_reversal_ids)
)
clean = clean.drop(columns="row_id").reset_index(drop=True)

summary = pd.DataFrame(
    {
        "pattern": ["reversed pair (purchase leg)", "reversed pair (reversal leg)",
                    "multi-swipe follow-on", "TOTAL removed"],
        "rows": [len(reversed_purchase_ids), len(reversed_reversal_ids),
                 len(multiswipe_ids), len(drop_ids)],
        "dollars": [round(reversed_dollars, 2), round(reversed_dollars, 2),
                    round(multiswipe_dollars, 2), None],
    }
)
print(summary.to_string(index=False))
print()
print(f"Rows before : {N_START:,}")
print(f"Rows after  : {len(clean):,}  ({len(clean) / N_START:.1%} kept)")
print(f"Fraud rate before : {df['isFraud'].mean():.4%}")
print(f"Fraud rate after  : {clean['isFraud'].mean():.4%}")
print(f"Fraud rows dropped as duplicates : {int(df.loc[df['row_id'].isin(drop_ids), 'isFraud'].sum())}")


                     pattern  rows    dollars
reversed pair (purchase leg) 17758 2666519.27
reversed pair (reversal leg) 17758 2666519.27
       multi-swipe follow-on  7259 1071670.25
               TOTAL removed 42775        NaN

Rows before : 786,363
Rows after  : 743,588  (94.6% kept)
Fraud rate before : 1.5790%
Fraud rate after  : 1.5714%
Fraud rows dropped as duplicates : 732


In [10]:
OUT_PATH = PROCESSED_DIR / "transactions_deduped.parquet"
clean.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH.relative_to(REPO_ROOT)} - {OUT_PATH.stat().st_size:,} bytes, {len(clean):,} rows")
clean.head(3)


Wrote data\processed\transactions_deduped.parquet - 26,625,157 bytes, 743,588 rows


,accountNumber,customerId,creditLimit,availableMoney,transactionDateTime,transactionAmount,merchantName,acqCountry,merchantCountryCode,posEntryMode,posConditionCode,merchantCategoryCode,currentExpDate,accountOpenDate,dateOfLastAddressChange,cardCVV,enteredCVV,cardLast4Digits,transactionType,echoBuffer,currentBalance,merchantCity,merchantState,merchantZip,cardPresent,posOnPremises,recurringAuthInd,expirationDateKeyInMatch,isFraud,reversalUnmatched
0,419104777,419104777,50000,50000.0,2016-01-01 00:01:02,44.09,Washington Post,US,US,09,01,subscriptions,03/2028,2015-05-30,2015-05-30,837,837,5010,PURCHASE,,0.0,,,,False,,,False,False,False
1,674577133,674577133,5000,5000.0,2016-01-01 00:01:44,329.57,staples.com,US,US,09,08,online_retail,10/2024,2015-08-19,2015-08-19,430,430,1693,PURCHASE,,0.0,,,,False,,,False,False,False
2,958438658,958438658,20000,20000.0,2016-01-01 00:01:47,164.57,cheapfast.com,US,US,05,01,online_retail,04/2023,2013-07-20,2013-07-20,445,445,2062,PURCHASE,,0.0,,,,False,,,False,False,False


## Business impact, in plain terms

- **Reversed transactions overstate sales by about $2.67 million.** Roughly
  17,800 purchases in this year of data were reversed. If revenue is totalled
  straight off the raw log, every one of those is counted as a sale that never
  really happened. Netting them out is the correct treatment.
- **Multi-swipe errors add about $1.07 million of phantom charges.** Around
  7,300 charges are a terminal ringing up the same sale two or more times within
  five minutes. Left in, they inflate both sales figures and the apparent
  purchase frequency of about 1,900 accounts, and in the real world they are the
  kind of thing that generates customer complaints and chargebacks.
- **Together these are about 5% of all rows.** Removing them barely moves the
  fraud rate (1.58% to 1.57%), so this cleanup is about accuracy of the revenue
  and behaviour picture, not about the fraud label.
- The cleaned dataset, `data/processed/transactions_deduped.parquet`, is what the
  modeling notebook uses.

**Next:** `03_modeling.ipynb` trains a baseline classifier on `isFraud` using
this deduped dataset.
